In [1]:
!pwd

/aiffel/aiffel/sentiment_classification/data


In [4]:
import pandas as pd

# 파일 경로
train_path = "/aiffel/aiffel/sentiment_classification/data/ratings_train (1).txt"
test_path = "/aiffel/aiffel/sentiment_classification/data/ratings_test (1).txt"

# 데이터 로드
train_data = pd.read_csv(train_path, sep='\t')
test_data = pd.read_csv(test_path, sep='\t')

# 데이터 크기 확인
print(f"Train 데이터 크기: {train_data.shape}")
print(f"Test 데이터 크기: {test_data.shape}")

# 중복 제거
train_data = train_data.drop_duplicates(subset=['document'])
test_data = test_data.drop_duplicates(subset=['document'])

# 결측값 제거
train_data = train_data.dropna()
test_data = test_data.dropna()

# 최종 데이터 크기 확인
print(f"전처리 후 Train 데이터 크기: {train_data.shape}")
print(f"전처리 후 Test 데이터 크기: {test_data.shape}")

# 데이터 확인
print(train_data.head())

# 데이터 저장 (선택 사항)
train_data.to_csv("/aiffel/aiffel/sentiment_classification/data/ratings_train_cleaned.csv", index=False)
test_data.to_csv("/aiffel/aiffel/sentiment_classification/data/ratings_test_cleaned.csv", index=False)


Train 데이터 크기: (150000, 3)
Test 데이터 크기: (50000, 3)
전처리 후 Train 데이터 크기: (146182, 3)
전처리 후 Test 데이터 크기: (49157, 3)
         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1


### KoNLPy 형태소 분석기(Mecab, Okt, Komoran)

In [5]:
from konlpy.tag import Mecab, Okt, Komoran
import pandas as pd

# Mecab, Okt, Komoran 형태소 분석기 초기화
mecab = Mecab()
okt = Okt()
komoran = Komoran()

# 형태소 분석 함수
def tokenize_text(text, tokenizer):
    if isinstance(text, str):
        return ' '.join(tokenizer.morphs(text))
    return ''

# 데이터 불러오기
train_data = pd.read_csv("/aiffel/aiffel/sentiment_classification/data/ratings_train_cleaned.csv")
test_data = pd.read_csv("/aiffel/aiffel/sentiment_classification/data/ratings_test_cleaned.csv")

# 각 형태소 분석기를 적용한 컬럼 추가
for tokenizer, name in zip([mecab, okt, komoran], ['mecab', 'okt', 'komoran']):
    train_data[f'document_{name}'] = train_data['document'].apply(lambda x: tokenize_text(x, tokenizer))
    test_data[f'document_{name}'] = test_data['document'].apply(lambda x: tokenize_text(x, tokenizer))

# 저장
train_data.to_csv("/aiffel/aiffel/sentiment_classification/data/ratings_train_tokenized.csv", index=False)
test_data.to_csv("/aiffel/aiffel/sentiment_classification/data/ratings_test_tokenized.csv", index=False)

# 확인
print(train_data.head())


         id                                           document  label  \
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0   
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1   
2  10265843                                  너무재밓었다그래서보는것을추천한다      0   
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0   
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1   

                                      document_mecab  \
0                           아 더 빙 . . 진짜 짜증 나 네요 목소리   
1     흠 . .. 포스터 보고 초딩 영화 줄 . ... 오버 연기 조차 가볍 지 않 구나   
2                                너무 재 밓었다그래서보는것을추천한다   
3              교도소 이야기 구먼 . . 솔직히 재미 는 없 다 . . 평점 조정   
4  사이몬페그 의 익살 스런 연기 가 돋보였 던 영화 ! 스파이더맨 에서 늙 어 보이 ...   

                                        document_okt  \
0                               아 더빙 .. 진짜 짜증나네요 목소리   
1         흠 ... 포스터 보고 초딩 영화 줄 .... 오버 연기 조차 가볍지 않구나   
2                          너 무재 밓었 다그 래서 보는것을 추천 한 다   


### SentencePiece 모델 학습  
model_types = ["unigram", "bpe", "char", "word"]  
vocab_sizes = [8000, 16000]

In [12]:
import sentencepiece as spm
import os
import pandas as pd

# 경로 설정
spm_dir = "/aiffel/aiffel/sentiment_classification/sp_tokenizer/"
os.makedirs(spm_dir, exist_ok=True)
temp_file = os.path.join(spm_dir, "korean-english-park.train.ko.temp")

# Corpus 저장 (KoNLPy 전처리 없이 원본 텍스트 사용)
train_data = pd.read_csv("/aiffel/aiffel/sentiment_classification/data/ratings_train_cleaned.csv")
filtered_corpus = train_data['document'].dropna().tolist()

with open(temp_file, 'w', encoding='utf-8') as f:
    for row in filtered_corpus:
        f.write(str(row) + '\n')

# 모델 설정
model_types = ["unigram", "bpe", "char", "word"]
vocab_sizes = [8000, 16000]

# SentencePiece 모델 자동 학습
for model_type in model_types:
    for vocab_size in vocab_sizes:
        model_prefix = f"korean_spm_{model_type}_{vocab_size}"
        spm_cmd = f"--input={temp_file} --model_prefix={os.path.join(spm_dir, model_prefix)} \
                    --vocab_size={vocab_size} --character_coverage=0.9995 \
                    --model_type={model_type}"
        spm.SentencePieceTrainer.Train(spm_cmd)
        print(f"Model {model_prefix} trained successfully!")

# 저장된 모델 확인
os.system(f"ls -l {spm_dir}")


Model korean_spm_unigram_8000 trained successfully!
Model korean_spm_unigram_16000 trained successfully!
Model korean_spm_bpe_8000 trained successfully!
Model korean_spm_bpe_16000 trained successfully!
Model korean_spm_char_8000 trained successfully!
Model korean_spm_char_16000 trained successfully!
Model korean_spm_word_8000 trained successfully!
Model korean_spm_word_16000 trained successfully!
total 17404
-rw-r--r-- 1 root root 13067962 Feb 25 07:12 korean-english-park.train.ko.temp
-rw-r--r-- 1 root root   523270 Feb 25 07:14 korean_spm_bpe_16000.model
-rw-r--r-- 1 root root   258384 Feb 25 07:14 korean_spm_bpe_16000.vocab
-rw-r--r-- 1 root root   370354 Feb 25 07:13 korean_spm_bpe_8000.model
-rw-r--r-- 1 root root   115472 Feb 25 07:13 korean_spm_bpe_8000.vocab
-rw-r--r-- 1 root root   258167 Feb 25 07:14 korean_spm_char_16000.model
-rw-r--r-- 1 root root    21919 Feb 25 07:14 korean_spm_char_16000.vocab
-rw-r--r-- 1 root root   258166 Feb 25 07:14 korean_spm_char_8000.model
-rw-r

0

### SentencePiece를 이용한 텍스트 토큰화   
각 모델별 maxlen 차별화

In [17]:
import sentencepiece as spm
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_data = pd.read_csv("/aiffel/aiffel/sentiment_classification/data/ratings_train_cleaned.csv")
test_data = pd.read_csv("/aiffel/aiffel/sentiment_classification/data/ratings_test_cleaned.csv")

train_texts = train_data['document'].dropna().tolist()
test_texts = test_data['document'].dropna().tolist()

# 모든 SentencePiece 모델을 적용하여 데이터 변환
sp_tokenized_data = {}
maxlen_dict = {}  # 모델별 maxlen 저장

for model_type in ["unigram", "bpe", "char", "word"]:
    for vocab_size in [8000, 16000]:
        model_path = f"/aiffel/aiffel/sentiment_classification/sp_tokenizer/korean_spm_{model_type}_{vocab_size}.model"
        
        s = spm.SentencePieceProcessor()
        s.Load(model_path)
        
        # 문장 길이 분석
        train_lengths = [len(s.EncodeAsIds(sentence)) for sentence in train_texts]
        test_lengths = [len(s.EncodeAsIds(sentence)) for sentence in test_texts]
        
        # 상위 95% 커버하는 maxlen 설정
        maxlen_train = int(np.percentile(train_lengths, 95))
        maxlen_test = int(np.percentile(test_lengths, 95))
        maxlen = max(maxlen_train, maxlen_test)  # 두 개 중 더 긴 값 사용
        
        maxlen_dict[f"{model_type}_{vocab_size}"] = maxlen  # 모델별 maxlen 저장
        
        # 패딩 적용
        train_tensor = tf.keras.preprocessing.sequence.pad_sequences(
            [s.EncodeAsIds(sentence) for sentence in train_texts], padding='post', maxlen=maxlen
        )
        test_tensor = tf.keras.preprocessing.sequence.pad_sequences(
            [s.EncodeAsIds(sentence) for sentence in test_texts], padding='post', maxlen=maxlen
        )
        
        key = f"{model_type}_{vocab_size}"
        sp_tokenized_data[key] = (train_tensor, test_tensor)
        
        print(f"{key} 데이터 변환 완료! maxlen: {maxlen}")

# 변환된 데이터 저장
np.save("/aiffel/aiffel/sentiment_classification/sp_tokenized_data.npy", sp_tokenized_data)
np.save("/aiffel/aiffel/sentiment_classification/sp_maxlen.npy", maxlen_dict)  # maxlen 저장

# 모델별 maxlen 확인
print("\n==== 모델별 maxlen 설정 ====")
for key, maxlen in maxlen_dict.items():
    print(f"{key}: maxlen = {maxlen}")


unigram_8000 데이터 변환 완료! maxlen: 52
unigram_16000 데이터 변환 완료! maxlen: 46
bpe_8000 데이터 변환 완료! maxlen: 51
bpe_16000 데이터 변환 완료! maxlen: 45
char_8000 데이터 변환 완료! maxlen: 110
char_16000 데이터 변환 완료! maxlen: 110
word_8000 데이터 변환 완료! maxlen: 18
word_16000 데이터 변환 완료! maxlen: 19

==== 모델별 maxlen 설정 ====
unigram_8000: maxlen = 52
unigram_16000: maxlen = 46
bpe_8000: maxlen = 51
bpe_16000: maxlen = 45
char_8000: maxlen = 110
char_16000: maxlen = 110
word_8000: maxlen = 18
word_16000: maxlen = 19


### RNN 모델 자동 학습 및 평가  

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
import numpy as np

# 레이블 데이터 불러오기
train_labels = pd.read_csv("/aiffel/aiffel/sentiment_classification/data/ratings_train_cleaned.csv")["label"].values
test_labels = pd.read_csv("/aiffel/aiffel/sentiment_classification/data/ratings_test_cleaned.csv")["label"].values

# 하이퍼파라미터 설정
embedding_dim = 8
hidden_units = 4
dropout_rate = 0.3
batch_size = 512
epochs = 3  # 빠른 테스트 위해 5로 설정

# 변환된 데이터 로드
sp_tokenized_data = np.load("/aiffel/aiffel/sentiment_classification/sp_tokenized_data.npy", allow_pickle=True).item()

# RNN 모델 학습 및 평가 자동화
results_dict = {}

for key, (train_tensor, test_tensor) in sp_tokenized_data.items():
    vocab_size = 8000 if "8000" in key else 16000
    
    # 모델 정의
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True),
        Bidirectional(LSTM(hidden_units, return_sequences=True)),
        Dropout(dropout_rate),
        Bidirectional(LSTM(hidden_units)),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])

    # 모델 컴파일
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    # 모델 학습
    history = model.fit(
        train_tensor, train_labels,
        validation_data=(test_tensor, test_labels),
        batch_size=batch_size,
        epochs=epochs,
        verbose=1
    )

    # 테스트 성능 평가
    test_loss, test_acc = model.evaluate(test_tensor, test_labels, verbose=1)
    results_dict[key] = test_acc
    print(f"{key} 모델 평가 완료! Accuracy: {test_acc:.4f}")

# 결과 저장
np.save("/aiffel/aiffel/sentiment_classification/sp_results.npy", results_dict)


Epoch 1/3
286/286 [==============================] - 172s 489ms/step - loss: 0.5223 - accuracy: 0.7625 - val_loss: 0.3892 - val_accuracy: 0.8409
Epoch 2/3
286/286 [==============================] - 134s 468ms/step - loss: 0.3658 - accuracy: 0.8540 - val_loss: 0.3626 - val_accuracy: 0.8487
Epoch 3/3
1537/1537 [==============================] - 36s 24ms/step - loss: 0.3560 - accuracy: 0.8490
unigram_8000 모델 평가 완료! Accuracy: 0.8490
Epoch 1/3
286/286 [==============================] - 161s 470ms/step - loss: 0.5260 - accuracy: 0.7582 - val_loss: 0.3841 - val_accuracy: 0.8463
Epoch 2/3
286/286 [==============================] - 118s 413ms/step - loss: 0.3425 - accuracy: 0.8696 - val_loss: 0.3529 - val_accuracy: 0.8537
Epoch 3/3
1537/1537 [==============================] - 34s 22ms/step - loss: 0.3482 - accuracy: 0.8539
unigram_16000 모델 평가 완료! Accuracy: 0.8539
Epoch 1/3
286/286 [==============================] - 194s 506ms/step - loss: 0.5199 - accuracy: 0.7626 - val_loss: 0.3877 - val_accur

### 최종 결과 비교 및 시각화  

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 결과 로드
results_dict = np.load("/aiffel/aiffel/sentiment_classification/sp_results.npy", allow_pickle=True).item()

# 데이터 변환
model_keys = list(results_dict.keys())
accuracies = list(results_dict.values())

# 성능 비교 테이블 출력
print("==== 모델별 성능 비교 ====")
for key, acc in results_dict.items():
    print(f"{key}: Accuracy {acc:.4f}")

# 성능 비교 시각화
plt.figure(figsize=(12, 6))
plt.barh(model_keys, accuracies, color='skyblue')
plt.xlabel("Accuracy")
plt.ylabel("SentencePiece Model")
plt.title("SentencePiece Model Performance Comparison")
plt.xlim(0.8, 1.0)  # Accuracy 범위 설정
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()


### 한계점 및 개선 방향  
모델 아키텍처 변경 실험 필요  
현재는 기본 RNN(LSTM) 모델을 사용했지만, 성능 향상을 위해 GRU, CNN, Transformer 기반 모델을 실험해볼 필요가 있다.  

더 다양한 SentencePiece 설정 실험 필요  
이번 실험에서는 character_coverage=0.9995를 고정했지만, 이를 조절하여 더 다양한 단어 분포를 반영할 수도 있다.  

추가적인 데이터 정제 및 증강 필요  
현재는 간단한 전처리만 수행했지만, 맞춤법 교정, 불용어 제거, 데이터 증강 등의 추가적인 정제 과정을 적용하면 성능을 향상시킬 수 있을 것이다.  

KoNLPy와 SentencePiece의 결합 실험 가능  
SentencePiece 단독 사용과 KoNLPy 단독 사용을 비교했지만, 형태소 분석 후 SentencePiece를 적용하는 하이브리드 방식도 고려해볼 수 있다.  